# Chapter 8 companion: open-loop sampled-data systems.

*Computers in the Loop: Digital Control from Sampling to Networked Agents*: EECE 5610 lecture notes prepared and revised by Milad Siami.

Course sources: Mario Sznaier's lecture notes; T. Charalambous's slides; Phillips, Nagle and Chakrabortty, *Digital Control System Analysis and Design*, 4th ed., Pearson (2015).

Reproduces:
  * E(s) = 1/((s+1)(s+2)), T = 0.5 s: E(z) = 0.239 z/((z - 0.607)(z - 0.368)),
    from the samples and from the residues;
  * ZOH + 1/(s+1): the pulse transfer function (1 - e^-T)/(z - e^-T), by brute
    force (integrating the plant over one period) and by an exact state-space
    discretization; step samples 0, 0.393, 0.632, 0.777 (T = 0.5 s); DC gain G(1);
  * where the samplers sit: G1 = 1/s, G2 = 1/(s+1), impulse input,
    (a) a sampler between the blocks, (b) no sampler between them;
  * the digital filter m(k) = 2e(k) - e(k-1) in front of ZOH + 1/(s+1);
  * how large T may be: ZOH equivalents of 25/(s^2 + 2s + 25) for T = 1 s and
    T = 0.1 s, and the true peak 1.527 against the largest sample.

Run the cells from top to bottom (in Colab: *Runtime > Run all*). Figures are also saved to a `figures` folder.


In [ ]:
from pathlib import Path

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from scipy import signal
from scipy.linalg import expm

FIG = Path.cwd() / "figures"
FIG.mkdir(exist_ok=True)

In [ ]:
def zoh(num, den, T):
    """ZOH equivalent of num/den: Phi, Gamma from one matrix exponential, then
    G(z) = C (zI - Phi)^-1 Gamma + D as numerator/denominator in powers of z."""
    A, B, C, D = signal.tf2ss(num, den)
    n = A.shape[0]
    M = np.zeros((n + 1, n + 1))
    M[:n, :n], M[:n, n:] = A, B
    E = expm(M * T)
    Phi, Gam = E[:n, :n], E[:n, n:]
    numz, denz = signal.ss2tf(Phi, Gam, C, D)
    return np.trim_zeros(np.round(numz[0], 14), "f"), denz

In [ ]:
def sg(v, d=4):
    """' + 0.1234' or ' - 0.1234' for printing polynomials."""
    return f" {'-' if v < 0 else '+'} {abs(v):.{d}f}"

In [ ]:
# ---------------------------------------------------------------- E(z) of a sampled signal
T = 0.5

In [ ]:
a, b = np.exp(-T), np.exp(-2 * T)
k = np.arange(8)
print("E(s) = 1/((s+1)(s+2)), T = 0.5 s")
print(f"  E(z) = {a - b:.3f} z/((z - {a:.3f})(z - {b:.3f}))")
series = signal.lfilter([0, a - b], np.polymul([1, -a], [1, -b]), np.r_[1.0, np.zeros(7)])
print("  samples e^-kT - e^-2kT :", np.round(np.exp(-k * T) - np.exp(-2 * k * T), 4))
print("  series of E(z)         :", np.round(series, 4))

In [ ]:
# ---------------------------------------------------------------- ZOH + 1/(s+1)
numz, denz = zoh([1], [1, 1], T)

In [ ]:
print("\nZOH + 1/(s+1), T = 0.5 s")
print(f"  exact discretization: G(z) = {numz[-1]:.4f}/(z{sg(denz[1])});  (1 - e^-T)/(z - e^-T) = "
      f"{1 - a:.4f}/(z - {a:.4f})")
c = np.zeros(6)

In [ ]:
for kk in range(5):                      # brute force: integrate the plant over one period
    c[kk + 1] = a * c[kk] + (1 - a) * 1.0

In [ ]:
print("  step samples (brute force):", np.round(c, 3), "  1 - e^-kT:", np.round(1 - np.exp(-np.arange(6) * T), 3))
print(f"  DC gain G(1) = {np.polyval(numz, 1) / np.polyval(denz, 1):.4f} = G_p(0)")

In [ ]:
# ---------------------------------------------------------------- configurations (a) and (b)
imp = np.r_[1.0, np.zeros(5)]

In [ ]:
ca = signal.lfilter([1, 0, 0], np.polymul([1, -1], [1, -a]), imp)          # G1(z) G2(z)
cb = signal.lfilter([0, 1 - a, 0], np.polymul([1, -1], [1, -a]), imp)      # G1G2(z)
print("\nG1 = 1/s, G2 = 1/(s+1), T = 0.5 s, unit impulse sequence in")
print("  (a) sampler between the blocks, G1(z)G2(z):", np.round(ca[:4], 3))
print("  (b) no sampler between them,    G1G2(z)   :", np.round(cb[:4], 3))

In [ ]:
# ---------------------------------------------------------------- digital filter
print("\nDigital filter D(z) = (2z - 1)/z before ZOH + 1/(s+1), unit step, T = 0.5 s")

In [ ]:
m = signal.lfilter([2, -1], [1], np.ones(8))                # m(k) = 2e(k) - e(k-1)
cf = signal.lfilter([0, 1 - a], [1, -a], m)
print("  with the filter   :", np.round(cf[:5], 3), "  formula 1 - (2a-1) a^(k-1):",
      np.round(np.r_[0, 1 - (2 * a - 1) * a ** (np.arange(1, 5) - 1)], 3))
print("  without it (D = 1):", np.round(signal.lfilter([0, 1 - a], [1, -a], np.ones(5)), 3))

In [ ]:
# ---------------------------------------------------------------- how large can T be?
num, den = [25], [1, 2, 25]

In [ ]:
tt = np.linspace(0, 9, 90001)
_, yc = signal.step((num, den), T=tt)
print("\nG_p = 25/(s^2 + 2s + 25): zeta = 0.2, wn = 5 rad/s")
print(f"  continuous step: peak {yc.max():.3f} at t = {tt[yc.argmax()]:.2f} s,"
      f" period 2 pi/wd = {2 * np.pi / (5 * np.sqrt(1 - 0.04)):.2f} s")
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))

In [ ]:
for ax, T in zip(axes, (1.0, 0.1)):
    numz, denz = zoh(num, den, T)
    n = int(round((6 if T < 0.5 else 9) / T)) + 1
    cs = signal.lfilter(np.r_[0, numz], denz, np.ones(n))
    print(f"  T = {T}: G(z) = ({numz[0]:.4f} z{sg(numz[1])})/(z^2{sg(denz[1])} z{sg(denz[2])}),"
          f" G(1) = {np.polyval(numz, 1) / np.polyval(denz, 1):.4f}")
    print(f"          samples {np.round(cs[:4], 3)} ...; largest sample {cs.max():.3f}"
          f" at k = {cs.argmax()} (t = {cs.argmax() * T:.1f} s)")
    m_ = tt <= (n - 1) * T
    ax.plot(tt[m_], yc[m_], color="gray", label="c(t)")
    ax.plot(np.arange(n) * T, cs, "o", color="C3", ms=4, label="c(kT)")
    ax.set(xlabel="t (s)", title=f"T = {T} s")
    ax.legend(fontsize=8)

In [ ]:
fig.tight_layout()
fig.savefig(FIG / "ch08_how_large_T.png", dpi=150)
print("\nSaved figures/ch08_how_large_T.png")

## Explore




In [ ]:
# In the browser (JupyterLite) the sliders need one extra package.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install("ipywidgets")

In [ ]:
r"""**How large can $T$ be?** A unit step goes through the zero-order hold into
$G_p(s) = 25/(s^2 + 2\zeta\cdot5\,s + 25)$, with $\zeta = 0.2$ by default: the true output
$c(t)$ peaks at 1.527 at $t = 0.64$ s. The samples $c(kT)$ come from the pulse transfer
function $G(z) = (1 - z^{-1})\,\mathcal{Z}[G_p(s)/s]$ and are exact, but they show only what
happens at $t = kT$. Start at $T = 1$ s (about 1.3 samples per oscillation, the peak is
missed), then shrink $T$ toward 0.1 s and watch the samples trace the response. Watch
the poles $z = e^{sT}$ move around the circle too.
"""
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.linalg import expm

plt.rcParams.update({"font.size": 11})
ORANGE, BLUE, TEAL, GRAY = "#E67E22", "#1D5FA8", "#1F6E78", "#6B7785"
WN = 5.0
T_END = 6.0


def _zoh(zeta, T):
    """ZOH equivalent of WN^2/(s^2 + 2 zeta WN s + WN^2): G(z) = num/den in powers of z."""
    A = np.array([[0.0, 1.0], [-WN ** 2, -2 * zeta * WN]])
    B = np.array([[0.0], [WN ** 2]])
    C = np.array([[1.0, 0.0]])
    M = np.zeros((3, 3))
    M[:2, :2], M[:2, 2:] = A, B
    E = expm(M * T)
    Phi, Gam = E[:2, :2], E[:2, 2:]
    num, den = signal.ss2tf(Phi, Gam, C, np.zeros((1, 1)))
    return num[0][1:], den          # num = b1 z + b2


def explore(T=1.0, zeta=0.2):
    T, zeta = round(T, 10), round(zeta, 10)
    num, den = _zoh(zeta, T)
    n = int(np.floor(T_END / T)) + 1
    c_k = signal.lfilter(np.r_[0.0, num], den, np.ones(n))      # unit step samples
    t = np.linspace(0, T_END, 2000)
    _, c_t = signal.step(([WN ** 2], [1.0, 2 * zeta * WN, WN ** 2]), T=t)
    poles = np.roots(den)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 4.0),
                                   gridspec_kw={"width_ratios": [1.9, 1]})
    ax1.plot(t, c_t, color=GRAY, lw=1.6, label="c(t)")
    ax1.plot(np.arange(n) * T, c_k, "o", color=ORANGE, ms=6, label="samples c(kT)")
    ax1.axhline(1, color="k", lw=0.6)
    ax1.set_xlim(0, T_END)
    ax1.set_ylim(0, max(1.7, c_t.max() + 0.15))
    ax1.set_xlabel("t (s)")
    ax1.set_ylabel("c")
    ax1.set_title(rf"Unit step through the ZOH: T = {T:g} s, $\zeta$ = {zeta:g}", fontsize=11)
    ax1.legend(fontsize=10, loc="lower right")

    th = np.linspace(0, 2 * np.pi, 300)
    ax2.plot(np.cos(th), np.sin(th), color=GRAY, lw=1)
    ax2.axhline(0, color="k", lw=0.5)
    ax2.axvline(0, color="k", lw=0.5)
    ax2.plot(poles.real, poles.imag, "x", color=ORANGE, ms=12, mew=2.5,
             label=r"poles $e^{s_i T}$")
    ax2.set_xlim(-1.3, 1.3)
    ax2.set_ylim(-1.3, 1.3)
    ax2.set_aspect("equal")
    ax2.set_xlabel("Re z")
    ax2.set_ylabel("Im z")
    ax2.set_title("poles of G(z)", fontsize=11)
    ax2.legend(fontsize=10, loc="upper left")
    fig.tight_layout()

    sg = lambda v: f"{'-' if v < 0 else '+'} {abs(v):.4f}"
    print(f"G(z) = ({num[0]:.4f} z {sg(num[1])}) / (z^2 {sg(den[1])} z {sg(den[2])}),"
          f"  G(1) = {np.sum(num) / np.sum(den):.4f}")
    i = int(np.argmax(c_k))
    peak = (f"true peak {c_t.max():.3f} at t = {t[np.argmax(c_t)]:.2f} s" if c_t.max() > 1.001
            else "c(t) rises without overshoot")
    print(f"{peak};  largest sample c({i}) = {c_k[i]:.3f} at t = {i * T:g} s")
    if zeta < 1:
        wd = WN * np.sqrt(1 - zeta ** 2)
        print(f"oscillation period 2 pi/wd = {2 * np.pi / wd:.2f} s: "
              f"{2 * np.pi / (wd * T):.1f} samples per period")
    else:
        print("no oscillation (zeta >= 1)")
    plt.show()


try:
    from ipywidgets import interact, FloatSlider
    interact(explore,
             T=FloatSlider(value=1.0, min=0.05, max=1.5, step=0.05, description="T (s)",
                           continuous_update=False),
             zeta=FloatSlider(value=0.2, min=0.05, max=1.0, step=0.05, description="zeta",
                              continuous_update=False))
except ImportError:
    explore()

## Hands-on

**Hands-on, Chapter 8.** These cells go with the Hands-on exercises at the end of
Chapter 8. Do each exercise on paper first and write down your prediction; then use the
matching section below to check it. Each section has sliders: change one thing at a time.


In [ ]:
# In the browser (JupyterLite) the sliders need one extra package.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install("ipywidgets")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

plt.rcParams.update({"font.size": 11})
ORANGE, BLUE, TEAL, GRAY = "#E67E22", "#1D5FA8", "#1F6E78", "#6B7785"

In [ ]:
def held_response(num, den, u, T, m=200):
    """Continuous response of num/den to the held sequence u (period T), m points per period."""
    t = np.arange(len(u) * m) * (T / m)
    _, y, _ = signal.lsim(signal.lti(num, den), np.repeat(u, m), t, interp=False)
    return t, y

In [ ]:
# ---------------------------------------------------------------- H1
def h1_double_integrator(T=0.1, command="alternating (-1)^k"):
    """H1: held acceleration command into 1/s^2; altitude y(t) and its samples."""
    n = 20
    k = np.arange(n)
    u = (-1.0) ** k if command.startswith("alt") else np.ones(n)
    t, y = held_response([1.0], [1.0, 0, 0], u, T)
    num, den, _ = signal.cont2discrete(([1.0], [1.0, 0, 0]), T, method="zoh")
    zeros = np.roots(np.trim_zeros(np.ravel(num), "f"))
    fig, ax = plt.subplots(figsize=(10, 3.6))
    ax.plot(t, y, color=GRAY, lw=1.4, label="y(t)")
    ax.plot(k * T, y[::200], "o", color=ORANGE, ms=6, label="samples y(kT)")
    ax.set_xlabel("t (s)")
    ax.set_ylabel("m")
    ax.legend(loc="upper left", fontsize=9)
    ax.set_title(f"T = {T:g} s, {command}: zero of G(z) at {zeros[0].real:+.3f}", fontsize=11)
    plt.show()

In [ ]:
# ---------------------------------------------------------------- H2
def h2_hold_or_impulses(T=0.3466):
    """H2: belt 3/(s+2): ZOH equivalent vs the impulse-driven model Z[Gp], unit step."""
    p = np.exp(-2 * T)
    k = np.arange(12)
    right = 1.5 * (1 - p ** k)                      # (z-1)/z Z[3/(s(s+2))]
    wrong = 3 * (1 - p ** (k + 1)) / (1 - p)        # Z[3/(s+2)] times z/(z-1)
    t, y = held_response([3.0], [1.0, 2.0], np.ones(len(k)), T)
    fig, ax = plt.subplots(figsize=(10, 3.6))
    ax.plot(t, y, color=GRAY, lw=1.4, label="belt speed c(t), held 1 V step")
    ax.plot(k * T, right, "o", color=ORANGE, ms=7, label="(z-1)/z Z[Gp/s]")
    ax.plot(k * T, wrong, "s--", color=TEAL, ms=5, lw=1, label="Gho(z) Gp(z)")
    ax.set_xlabel("t (s)")
    ax.set_ylabel("m/s")
    ax.legend(loc="center right", fontsize=9)
    ax.set_title(f"T = {T:.3f} s: DC gain 1.5 (right) vs {3 / (1 - p):.2f} (slip)", fontsize=11)
    plt.show()

In [ ]:
# ---------------------------------------------------------------- H3
def h3_crane(T=1.5):
    """H3: lightly damped crane payload (2 s swing) and the samples of its step response."""
    sig = np.log(2) / 2
    wn2 = sig ** 2 + np.pi ** 2
    tt = np.linspace(0, 12, 3000)
    c = lambda t: 1 - np.exp(-sig * t) * (np.cos(np.pi * t) + sig / np.pi * np.sin(np.pi * t))
    k = np.arange(int(40 / T) + 1)                # 40 s of samples for the test below
    ck = c(k * T)
    mono = bool(np.all(np.diff(ck) >= -1e-12) and np.all(ck <= 1 + 1e-12))
    poles = np.exp(np.array([-sig + 1j * np.pi, -sig - 1j * np.pi]) * T)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 3.8), gridspec_kw={"width_ratios": [2, 1]})
    ax1.plot(tt, c(tt), color=GRAY, lw=1.2, label="payload c(t)")
    ax1.plot(k * T, ck, "o-", color=ORANGE, ms=6, lw=1, label="samples")
    ax1.axhline(1, color="k", lw=0.6)
    ax1.set_ylim(-0.1, 1.9)
    ax1.set_xlabel("t (s)")
    ax1.legend(loc="upper right", fontsize=9)
    ax1.set_xlim(0, 12)
    ax1.set_title(f"T = {T:g} s: largest sample {ck.max():.3f} (true peak {c(1.0):.3f}), "
                  f"monotone: {'yes' if mono else 'no'}", fontsize=10)
    th = np.linspace(0, 2 * np.pi, 300)
    ax2.plot(np.cos(th), np.sin(th), color=GRAY, lw=1)
    ax2.plot(poles.real, poles.imag, "x", color=ORANGE, ms=12, mew=2.5)
    ax2.set_aspect("equal")
    ax2.set_xlim(-1.3, 1.3)
    ax2.set_ylim(-1.3, 1.3)
    ax2.set_title(f"poles of G(z), wn^2 = {wn2:.2f}", fontsize=10)
    plt.show()

In [ ]:
# ---------------------------------------------------------------- H4
def h4_delay_line(n=6):
    """H4: dryer 2 e^{-0.6 s}/(s+1) with T = 0.6/n; step response of the sampled model."""
    T = 0.6 / n
    p = np.exp(-T)
    kmax = int(np.ceil(5 / T))
    k = np.arange(kmax + 1)
    c = np.where(k >= n + 1, 2 * (1 - p ** (k - n)), 0.0)
    tt = np.linspace(0, kmax * T, 2000)
    fig, ax = plt.subplots(figsize=(10, 3.6))
    ax.plot(tt, np.where(tt > 0.6, 2 * (1 - np.exp(-(tt - 0.6))), 0), color=GRAY, lw=1.4,
            label="c(t)")
    ax.plot(k * T, c, "o", color=ORANGE, ms=4, label="samples of the model")
    ax.set_xlabel("t (s)")
    ax.set_ylabel("deg C per W")
    ax.legend(loc="lower right", fontsize=9)
    ax.set_title(f"n = {n}, T = {T:.4f} s: G(z) = {2 * (1 - p):.3f} / (z^{n} (z - {p:.3f})), "
                 f"first nonzero c({n + 1}) = {c[n + 1]:.3f}, G(1) = 2", fontsize=10)
    plt.show()

In [ ]:
try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown
    interact(h1_double_integrator,
             T=FloatSlider(value=0.1, min=0.05, max=0.5, step=0.05, description="T (s)",
                           continuous_update=False),
             command=Dropdown(options=["alternating (-1)^k", "step"], value="alternating (-1)^k",
                              description="command"))
    interact(h2_hold_or_impulses, T=FloatSlider(value=0.3466, min=0.05, max=1.0, step=0.01,
                                                description="T (s)", continuous_update=False))
    interact(h3_crane, T=FloatSlider(value=1.5, min=0.5, max=2.5, step=0.05, description="T (s)",
                                     continuous_update=False))
    interact(h4_delay_line, n=IntSlider(value=6, min=1, max=12, description="n = 0.6/T",
                                        continuous_update=False))
except ImportError:
    h1_double_integrator()
    h2_hold_or_impulses()
    h3_crane()
    h4_delay_line()